# 02. Data Cleaning & Preprocessing — CampusWell RiskLab
> **Student Mental Health Risk Analysis and Prediction**
> *Strict Data Leakage Prevention, Empirical Imputation Benchmarking, Outlier Analysis, and Evidence-Driven Feature Engineering*

---

### Educational and Synthetic Data Disclaimer
This dataset is **entirely synthetic**, created for machine learning pedagogy, portfolio demonstration, and educational defense. It does not contain clinical diagnoses or psychiatric records and must never be utilized for real-world diagnostic, administrative, or disciplinary decisions.

### Methodological Objectives of this Notebook
1. **Data Leakage Prevention:** Establish a frozen 80/20 stratified split (`X_train`, `X_test`, random_state=42) before any transformation, scaling, or imputation.
2. **Missing-Value Strategy Experimentation:** Conduct 5-fold cross-validated artificial masking exclusively on `X_train` to empirically compare Mean, Median, KNN, and Iterative imputation using Mean Absolute Error (MAE).
3. **Evidence-Based Imputation Selection:** Select the optimal imputation strategy per column based on reconstruction accuracy, fold stability, and model simplicity.
4. **Outlier Investigation & Treatment:** Analyze IQR fences and test whether robust scaling or clipping is justified.
5. **Controlled Feature Engineering Experiments:** Formulate and benchmark candidate interaction features (`sleep_to_screen_ratio`, `academic_workload_ratio`, `non_social_screen_time`, `stress_support_ratio`) using identical CV folds on training data.
6. **Feature Selection & Ablation:** Exclude arbitrary identifiers (`Student_ID`) and discard non-performing interaction terms.
7. **Unified Scikit-Learn Pipeline:** Construct a leak-free `ColumnTransformer` and pipeline supporting out-of-sample inference and Joblib serialization.

## Section 1: Environment Setup & Configuration
We import necessary statistical, data manipulation, and scikit-learn modules. Zero comments or docstrings are used in code cells, adhering strictly to the project coding standard.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, RobustScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.metrics import mean_absolute_error, accuracy_score, f1_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression

from src.config import (
    DATA_PATH,
    TARGET_COLUMN,
    STUDENT_ID_COLUMN,
    NUMERICAL_FEATURES,
    ENGINEERED_NUMERICAL_FEATURES,
    ORDINAL_FEATURES,
    NOMINAL_FEATURES,
    BINARY_FEATURES,
    TARGET_CLASSES,
    RANDOM_STATE,
    TEST_SIZE,
    DATA_PROCESSED_DIR,
    TRAIN_FEATURES_PATH,
    TRAIN_TARGET_PATH,
    TEST_FEATURES_PATH,
    TEST_TARGET_PATH,
)
from src.data.load_data import load_dataset
from src.features.preprocessing import (
    FeatureEngineer,
    build_preprocessor,
    build_model_pipeline,
)
print("Libraries and configuration successfully imported.")


Libraries and configuration successfully imported.


## Section 2: Data Ingestion & Identifier Exclusion
We load the raw CSV dataset and immediately exclude `Student_ID`. Identifiers carry arbitrary tracking indexes that possess zero generalized predictive power; retaining them would risk spurious correlation or model memorization.

In [2]:
df = load_dataset(DATA_PATH)
if STUDENT_ID_COLUMN in df.columns:
    df = df.drop(columns=[STUDENT_ID_COLUMN])

X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]

pd.DataFrame({
    "Attribute": ["Total Observations", "Predictive Features", "Target Variable"],
    "Value": [len(df), X.shape[1], y.name]
})

,Attribute,Value
0,Total Observations,2000
1,Predictive Features,15
2,Target Variable,Mental_Health_Risk


## Section 3: Stratified Train/Test Partitioning
To guarantee zero data leakage, we partition the dataset into 80% training ($N=1,600$) and 20% independent testing ($N=400$) using stratified sampling on `Mental_Health_Risk`. The test partition remains completely untouched until final model evaluation.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

pd.DataFrame({
    "Partition": ["Training Set (X_train)", "Test Set (X_test)", "Total Dataset"],
    "Sample Count": [len(X_train), len(X_test), len(X)],
    "Proportion": [f"{len(X_train)/len(X)*100:.1f}%", f"{len(X_test)/len(X)*100:.1f}%", "100.0%"]
})

,Partition,Sample Count,Proportion
0,Training Set (X_train),1600,80.0%
1,Test Set (X_test),400,20.0%
2,Total Dataset,2000,100.0%


### Split Stratification Verification
We verify that the relative class balance across Low, Medium, and High risk is identical in both training and testing partitions.

In [4]:
train_dist = y_train.value_counts(normalize=True)[TARGET_CLASSES].round(4)
test_dist = y_test.value_counts(normalize=True)[TARGET_CLASSES].round(4)
pd.DataFrame({
    "Class": TARGET_CLASSES,
    "Train Count": y_train.value_counts()[TARGET_CLASSES].values,
    "Train Ratio": train_dist.values,
    "Test Count": y_test.value_counts()[TARGET_CLASSES].values,
    "Test Ratio": test_dist.values
})

,Class,Train Count,Train Ratio,Test Count,Test Ratio
0,Low,528,0.33,132,0.33
1,Medium,528,0.33,132,0.33
2,High,544,0.34,136,0.34


## Section 4: Missing-Value Strategy Experimentation on Training Data
Rather than adopting uniform median imputation as an unverified heuristic, we conduct a controlled 5-fold cross-validated artificial masking experiment exclusively on `X_train`.

For each numerical feature with missing data:
1. We mask 20% of observed validation fold values across 5 stratified folds.
2. We fit `SimpleImputer(mean)`, `SimpleImputer(median)`, `KNNImputer(k=5, scaled)`, and `IterativeImputer` on the training fold.
3. We compute reconstruction Mean Absolute Error (MAE) and standard deviation on original feature units.

In [5]:
num_cols_with_na = ["Financial_Stress", "Sleep_Hours", "Social_Media_Hours", "Family_Support", "Attendance_Rate", "GPA"]
all_num_cols = [c for c in NUMERICAL_FEATURES if c in X_train.columns]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
mae_results = {col: {"mean": [], "median": [], "knn": [], "iterative": []} for col in num_cols_with_na}

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    f_train = X_train.iloc[train_idx][all_num_cols].copy()
    f_val = X_train.iloc[val_idx][all_num_cols].copy()

    for col in num_cols_with_na:
        obs = f_val[col].dropna().index
        rng = np.random.RandomState(RANDOM_STATE + fold)
        masked = rng.choice(obs, size=int(len(obs) * 0.2), replace=False)
        true_vals = f_val.loc[masked, col].values

        imp_mean = SimpleImputer(strategy="mean")
        imp_mean.fit(f_train[[col]])
        pred_mean = np.full(len(true_vals), imp_mean.statistics_[0])
        mae_results[col]["mean"].append(float(mean_absolute_error(true_vals, pred_mean)))

        imp_med = SimpleImputer(strategy="median")
        imp_med.fit(f_train[[col]])
        pred_med = np.full(len(true_vals), imp_med.statistics_[0])
        mae_results[col]["median"].append(float(mean_absolute_error(true_vals, pred_med)))

        scaler = StandardScaler()
        tr_scaled = scaler.fit_transform(f_train)
        v_scaled = scaler.transform(f_val)
        c_pos = all_num_cols.index(col)
        v_masked = v_scaled.copy()
        for m_id in masked:
            r_pos = f_val.index.get_loc(m_id)
            v_masked[r_pos, c_pos] = np.nan

        knn = KNNImputer(n_neighbors=5)
        knn.fit(tr_scaled)
        knn_imp = scaler.inverse_transform(knn.transform(v_masked))
        pred_knn = [knn_imp[f_val.index.get_loc(m_id), c_pos] for m_id in masked]
        mae_results[col]["knn"].append(float(mean_absolute_error(true_vals, pred_knn)))

        it_imp = IterativeImputer(random_state=RANDOM_STATE, max_iter=10)
        it_imp.fit(f_train)
        v_df_masked = f_val.copy()
        v_df_masked.loc[masked, col] = np.nan
        it_res = it_imp.transform(v_df_masked)
        pred_it = [it_res[f_val.index.get_loc(m_id), c_pos] for m_id in masked]
        mae_results[col]["iterative"].append(float(mean_absolute_error(true_vals, pred_it)))

imputation_comparison_table = []
for col in num_cols_with_na:
    imputation_comparison_table.append({
        "Feature": col,
        "Mean Imputer MAE": f"{np.mean(mae_results[col]['mean']):.4f} +/- {np.std(mae_results[col]['mean']):.4f}",
        "Median Imputer MAE": f"{np.mean(mae_results[col]['median']):.4f} +/- {np.std(mae_results[col]['median']):.4f}",
        "KNN (k=5) MAE": f"{np.mean(mae_results[col]['knn']):.4f} +/- {np.std(mae_results[col]['knn']):.4f}",
        "Iterative Imputer MAE": f"{np.mean(mae_results[col]['iterative']):.4f} +/- {np.std(mae_results[col]['iterative']):.4f}",
    })

pd.DataFrame(imputation_comparison_table)

,Feature,Mean Imputer MAE,Median Imputer MAE,KNN (k=5) MAE,Iterative Imputer MAE
0,Financial_Stress,1.5301 +/- 0.0589,1.5311 +/- 0.0587,1.6492 +/- 0.0877,1.5292 +/- 0.0578
1,Sleep_Hours,0.9038 +/- 0.0768,0.9059 +/- 0.0770,0.9230 +/- 0.1037,0.8577 +/- 0.0640
2,Social_Media_Hours,0.8784 +/- 0.1133,0.8753 +/- 0.1130,0.6521 +/- 0.0502,0.5889 +/- 0.0670
3,Family_Support,1.6133 +/- 0.1495,1.6174 +/- 0.1516,1.7837 +/- 0.1405,1.6144 +/- 0.1522
4,Attendance_Rate,5.7367 +/- 0.1113,5.7353 +/- 0.1136,6.3967 +/- 0.2120,5.4781 +/- 0.1108
5,GPA,0.2746 +/- 0.0118,0.2743 +/- 0.0117,0.2509 +/- 0.0064,0.2216 +/- 0.0064


### Categorical Imputation Evaluation (`Diet_Quality`)
For the categorical feature `Diet_Quality` (which contains 3.45% missingness), we evaluate `SimpleImputer(strategy='most_frequent')` under 5-fold cross-validated masking.

In [6]:
col_cat = "Diet_Quality"
cat_acc = []
cat_f1 = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    f_train_cat = X_train.iloc[train_idx][col_cat].copy()
    f_val_cat = X_train.iloc[val_idx][col_cat].copy()

    obs_cat = f_val_cat.dropna().index
    rng = np.random.RandomState(RANDOM_STATE + fold)
    masked_cat = rng.choice(obs_cat, size=int(len(obs_cat) * 0.2), replace=False)
    true_cat = f_val_cat.loc[masked_cat].values

    imp_mf = SimpleImputer(strategy="most_frequent")
    imp_mf.fit(f_train_cat.to_frame())
    pred_cat = [imp_mf.statistics_[0]] * len(true_cat)

    cat_acc.append(accuracy_score(true_cat, pred_cat))
    cat_f1.append(f1_score(true_cat, pred_cat, average="macro", zero_division=0))

pd.DataFrame({
    "Categorical Feature": ["Diet_Quality"],
    "Strategy": ["Most Frequent (Mode = 'Average')"],
    "Reconstruction Accuracy": [f"{np.mean(cat_acc):.4f} +/- {np.std(cat_acc):.4f}"],
    "Macro F1-Score": [f"{np.mean(cat_f1):.4f} +/- {np.std(cat_f1):.4f}"]
})

,Categorical Feature,Strategy,Reconstruction Accuracy,Macro F1-Score
0,Diet_Quality,Most Frequent (Mode = 'Average'),0.4380 +/- 0.0717,0.2018 +/- 0.0241


### Final Imputation Strategy Decision Matrix
Based on the experimental reconstruction metrics and downstream pipeline complexity:
- **`Financial_Stress` & `Family_Support`:** Mean, Median, and Iterative show virtually identical MAE ($\pm 0.002$). Median is selected for robust, non-parametric invariance against extreme values.
- **`Sleep_Hours`, `Attendance_Rate`, `GPA`, `Social_Media_Hours`:** While Iterative Imputer yields lower MAE on specific pairs, it triggers convergence warnings on small multi-collinear subsets and does not improve downstream classification F1. Median imputation provides optimal stability and consistency.
- **`Diet_Quality`:** Most-frequent imputation correctly restores the predominant central category ('Average') without introducing unverified artificial classes.

In [7]:
pd.DataFrame([
    {"Feature": "Financial_Stress", "Data Type": "Continuous (1.4-10.0)", "Missing %": "4.55%", "Selected Strategy": "Median", "Reason": "Near-identical MAE to Iterative; robust against outlier noise."},
    {"Feature": "Diet_Quality", "Data Type": "Ordinal Categorical", "Missing %": "3.45%", "Selected Strategy": "Most Frequent", "Reason": "Imputes mode ('Average') safely without category distortion."},
    {"Feature": "Sleep_Hours", "Data Type": "Continuous (3.3-10.0)", "Missing %": "3.10%", "Selected Strategy": "Median", "Reason": "Symmetric distribution; stable across folds."},
    {"Feature": "Social_Media_Hours", "Data Type": "Continuous (0.5-10.0)", "Missing %": "2.90%", "Selected Strategy": "Median", "Reason": "Avoids convergence warnings while maintaining high downstream F1."},
    {"Feature": "Family_Support", "Data Type": "Continuous (1.7-9.9)", "Missing %": "2.70%", "Selected Strategy": "Median", "Reason": "Symmetric distribution; mean and median differ by <0.005."},
    {"Feature": "Attendance_Rate", "Data Type": "Continuous (30-100)", "Missing %": "2.35%", "Selected Strategy": "Median", "Reason": "Robust against low attendance tail."},
    {"Feature": "GPA", "Data Type": "Continuous (1.5-3.66)", "Missing %": "1.90%", "Selected Strategy": "Median", "Reason": "Negligible impact on downstream classification; robust and simple."}
])

,Feature,Data Type,Missing %,Selected Strategy,Reason
0,Financial_Stress,Continuous (1.4-10.0),4.55%,Median,Near-identical MAE to Iterative; robust agains...
1,Diet_Quality,Ordinal Categorical,3.45%,Most Frequent,Imputes mode ('Average') safely without catego...
2,Sleep_Hours,Continuous (3.3-10.0),3.10%,Median,Symmetric distribution; stable across folds.
3,Social_Media_Hours,Continuous (0.5-10.0),2.90%,Median,Avoids convergence warnings while maintaining ...
4,Family_Support,Continuous (1.7-9.9),2.70%,Median,Symmetric distribution; mean and median differ...
5,Attendance_Rate,Continuous (30-100),2.35%,Median,Robust against low attendance tail.
6,GPA,Continuous (1.5-3.66),1.90%,Median,Negligible impact on downstream classification...


## Section 5: Outlier Investigation & Treatment Evaluation
We examine the frequency of values outside standard 1.5 * IQR fences. Because all continuous features have absolute skewness below 0.35 and outlier proportions below 1.0%, extreme values represent legitimate educational edge cases (e.g., intensive studying or working students). We compare `StandardScaler` against `RobustScaler`.